# 1. Environment Setup and Library Installation
In this section, we install the necessary packages to work with Semantic Web technologies. We use `rdflib` for managing RDF graphs, `owlready2` for ontology handling, and `SPARQLWrapper` to connect to external databases.

In [ ]:
!pip install rdflib owlready2 requests pykeen SPARQLWrapper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 56.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 730.3/730.3 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.5 MB/s eta 0:00:00
  Created wheel for owlready2: filename=owlready2-0.50-cp312-cp312-linux_x86_64.whl size=24567084 sha256=721081be8706755c2f0cfff5504970f6a2f52535aa35fc00abb194498d6bb66c
  Stored in directory: /root/.cache/pip/wheels/fb/9d/9d/24582de3856fe0ff897408eb6ed1baa6cd5b7b112a2b9c711c
Successfully built owlready2

In [ ]:
import pandas as pd
from SPARQLWrapper import SPARQLWrapper, JSON
from owlready2 import *
import re
from numpy import integer

# 2. Extracting Local Data from Zaragoza Open Data
Here, we connect to the Zaragoza City Council's SPARQL endpoint. The goal is to collect local information about museums, parks, and transport systems and store it in a structured format using a Pandas DataFrame.

Query to access the api links from zaragoza website:

```
SELECT DISTINCT ?type WHERE {GRAPH <http://www.zaragoza.es/urbanismo-infraestructuras/equipamiento/recurso/> {?s a ?type}}
```


### Defining Structure

In [ ]:
# Columns of dataframe where we will create the triples from
ontology_columns = [
    'name', 'category', 'hasAddress', 'hasPrice', 'hasStars',
    'hasRating', 'hasWifi', 'hasParking', 'isPetFriendly',
    'hasStartDate', 'hasEndDate', 'accomodationType', 'hasForks'
]

In [ ]:
endpoint = "https://datos.zaragoza.es/sparql"
sparql = SPARQLWrapper(endpoint)

### SPARQL-to-Ontology Data Normalizer
This function converts raw SPARQL results into a clean format for your database. It maps data to your specific ontology columns.

In [ ]:
def execute_and_process(query, category_name):
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()

    processed_rows = []
    category = category_name.lower() #lowercasing to match easier

    for r in results["results"]["bindings"]:
        row = {col: None for col in ontology_columns}

        row['name'] = r["name"]["value"].title() if "name" in r else None
        row["name"] = row['name'].replace("`", "").replace("'", "") if row['name'] else None #removing punctuation marks that gave errors in protege
        row["category"] = category

        #hasAddress
        if "address" in r:
            row['hasAddress'] = r["address"]["value"]

        #Price
        if "price" in r:
            raw_p = r["price"]["value"].lower()
            if any(x in raw_p for x in ["gratuito", "gratuita", "gratis"]):
                row['hasPrice'] = 0.0
            else:
                match = re.search(r"(\d+[\.,]?\d*)", raw_p)
                if match:
                    row['hasPrice'] = float(match.group(1).replace(",", "."))

        #hasForks
        if "forks" in r:
            row["hasForks"] = r["forks"]["value"]

        # accomodationType - hasStars
        if "categoria" in r:
            cat = r["categoria"]["value"].lower()

            if "hotel" in cat:
                row['accomodationType'] = "Hotel"
                match = re.search(r"(\d+)", cat)
                if match:
                    row['hasStars'] = match.group(1)

            elif any(x in cat for x in ["v.u.t.", "apartamento turístico"]):
                row['accomodationType'] = "Apartment"

            elif "pensión" in cat or "hostal" in cat:
                row['accomodationType'] = "Hostal"
            else:
                continue

        # hasParking
        if "comment" in r:
            comment = r["comment"]["value"].lower()
            name = row['name'].lower() if row['name'] else ""

            if not "restaurante" in category:
              if "parking" in comment:
                  row['hasParking'] = True
              else:
                  row['hasParking'] = False

              if "animal" in comment:
                  row["isPetFriendly"] = True
              else:
                  row["isPetFriendly"] = False

              if "wifi" in comment:
                  row['hasWifi'] = True
              else:
                  row['hasWifi'] = False

            # Restaurant sub-categories based on the comment
            if "restaurante" in category:
                if "menú degustación" in comment or "vanguardia" in comment:
                    row["category"] = "fine_dining"
                elif any(w in (comment + " " + name) for w in ["tapas", "pintxos", "bar"]):
                    row["category"] = "tapas"
                elif "tradicional" in comment or name.startswith(("casa", "el ", "mesón", "taberna")):
                    row["category"] = "tradicional"
                elif any(w in (comment + " " + name) for w in ["burger", "hamburguesa", "kebab", "fast food"]):
                    row["category"] = "fast_food"
                elif any(w in (comment + " " + name) for w in ["panadería", "pastelería", "horno"]):
                    row["category"] = "bakery"

        processed_rows.append(row) #final df of data

    return processed_rows

### SPARQL Category Extractors
These functions use SPARQL queries to pull specific data for museums, transport, and parks from the Zaragoza endpoint. They retrieve names and optional details like addresses or prices, then send the results to a processor to be organized into the final table.

In [ ]:
# Kept the get_monument_data for museums too
# def get_museums_data():
#     query = """
#     PREFIX cultura: <http://www.zaragoza.es/api/kos/cultura-ocio/>
#     PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
#     PREFIX schema: <http://schema.org/>
#     PREFIX locn: <http://www.w3.org/ns/locn#>
#     SELECT DISTINCT ?name ?address ?price WHERE {
#       ?uri a cultura:museos ; rdfs:label ?name .
#       OPTIONAL { ?uri locn:fullAddress ?address . }
#       OPTIONAL { ?uri schema:price ?price . }
#     }
#     LIMIT 10
#     """
#     return execute_and_process(query, "museo")

def get_transport_data():
    query = """
    PREFIX transporte: <http://www.zaragoza.es/api/kos/transporte/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT DISTINCT ?name WHERE {
      ?uri a <http://www.zaragoza.es/api/kos/transporte/transporte> ;
           rdfs:label ?name .
    }

    """
    return execute_and_process(query, "transporte")

def get_parks_data():
    query = """
    PREFIX medio-ambiente: <http://www.zaragoza.es/api/kos/medio-ambiente/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX locn: <http://www.w3.org/ns/locn#>
    PREFIX schema: <http://schema.org/>

    SELECT DISTINCT ?name ?address WHERE {
      ?uri a medio-ambiente:parques-y-jardines ; rdfs:label ?name .

      OPTIONAL { ?uri locn:fullAddress ?address . }
      OPTIONAL { ?uri schema:streetAddress ?address . }
    }

    """
    return execute_and_process(query, "parque")

def get_galleries_data():
    query = """
    PREFIX cultura: <http://www.zaragoza.es/api/kos/cultura-ocio/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX schema: <http://schema.org/>
    PREFIX locn: <http://www.w3.org/ns/locn#>

    SELECT DISTINCT ?name ?address ?price WHERE {
      ?uri a cultura:galerias-de-arte ;
           rdfs:label ?name .

      OPTIONAL { ?uri locn:fullAddress ?address . }
      OPTIONAL { ?uri schema:price ?price . }
    }

    """
    return execute_and_process(query, "galeria")

def get_accomodation_data():
    query = """
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX vcard: <http://www.w3.org/2006/vcard/ns#>
    PREFIX aloj: <http://vocab.linkeddata.es/datosabiertos/def/turismo/alojamiento#>

    SELECT DISTINCT ?uri ?name ?address ?categoria ?comment WHERE {
      ?uri a <http://vocab.linkeddata.es/kos/turismo/alojamiento> .

      OPTIONAL { ?uri rdfs:label ?name }
      OPTIONAL { ?uri vcard:street-addr ?address }
      OPTIONAL { ?uri aloj:categoria ?categoria }
      OPTIONAL { ?uri rdfs:comment ?comment }
    }
    """
    return execute_and_process(query, "alojamientos")

def get_restaurants_data():
    query = """
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX locn: <http://www.w3.org/ns/locn#>

    SELECT DISTINCT ?uri ?name ?address ?comment ?forks WHERE {
      ?uri a <http://vocab.linkeddata.es/kos/turismo/restaurante> .

      OPTIONAL { ?uri rdfs:label ?name }
      OPTIONAL { ?uri locn:fullAddress ?address }
      OPTIONAL { ?uri rdfs:comment ?comment }

      OPTIONAL {
        ?uri <http://www.zaragoza.es/skos/vocab/tenedores> ?forks
      }
    }

    """
    return execute_and_process(query, "restaurante")

def get_monuments_data():
    query = """
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX locn: <http://www.w3.org/ns/locn#>
    PREFIX schema: <http://schema.org/>

    SELECT DISTINCT ?uri ?name ?address ?price WHERE {
      ?uri a <http://idi.fundacionctic.org/cruzar/turismo#Monumento> .
      OPTIONAL { ?uri rdfs:label ?name }
      OPTIONAL { ?uri locn:fullAddress ?address }
      OPTIONAL { ?uri schema:price ?price }
    }

    """
    return execute_and_process(query, "monumento")

### Data Aggregation and DataFrame Creation
This code combines the results from all individual category functions into a single dataframe.

In [ ]:
all_results = []

all_results.extend(get_transport_data())
all_results.extend(get_parks_data())
all_results.extend(get_galleries_data())
all_results.extend(get_accomodation_data())
all_results.extend(get_restaurants_data())
all_results.extend(get_monuments_data())

df = pd.DataFrame(all_results)
df.head()

,name,category,hasAddress,hasPrice,hasStars,hasRating,hasWifi,hasParking,isPetFriendly,hasStartDate,hasEndDate,accomodationType,hasForks
0,Estación De Cercanías Miraflores,transporte,None,NaN,None,None,None,None,None,None,None,None,None
1,Estación Central De Autobuses,transporte,None,NaN,None,None,None,None,None,None,None,None,None
2,Estación Zaragoza Delicias,transporte,None,NaN,None,None,None,None,None,None,None,None,None
3,Estación De Cercanias Portillo,transporte,None,NaN,None,None,None,None,None,None,None,None,None
4,Estación De Cercanías Goya,transporte,None,NaN,None,None,None,None,None,None,None,None,None


# 3. Retrieving Data from DBpedia
In this step, we connect to the mandatory SPARQL endpoint (DBpedia) to further expand our dataset.

In [ ]:
#DBPEDIA

sparql = SPARQLWrapper("https://dbpedia.org/sparql")
sparql.setReturnFormat(JSON)

query = """
PREFIX dbo: <http://dbpedia.org/ontology/>
PREFIX dbr: <http://dbpedia.org/resource/>

SELECT DISTINCT ?ind ?type WHERE {
  {
    dbr:Zaragoza dbo:wikiPageWikiLink ?ind .
  }
  UNION
  {
    dbr:List_of_tourist_attractions_in_Zaragoza dbo:wikiPageWikiLink ?ind .
  }

  ?ind rdf:type ?type .

  VALUES ?type {
    dbo:Castle
    dbo:PublicTransitSystem
    dbo:Holiday
  }
}
"""

sparql.setQuery(query)
results = sparql.query().convert()

data = [(r["ind"]["value"].split("/")[-1].replace("_", " "),
         r["type"]["value"].split("/")[-1])
        for r in results["results"]["bindings"]] #normalizing names that gave errors in protege

df_dbpedia = pd.DataFrame(data, columns=["name", "category"])

len(df_dbpedia)
df_dbpedia.head()

,name,category
0,Zaragoza tram,PublicTransitSystem
1,AVE,PublicTransitSystem
2,Aljafería,Castle
3,Cercanías Zaragoza,PublicTransitSystem
4,Fiestas del Pilar,Holiday


# 4. Ontology Population and Data Integration
This section initializes the RDF graph by loading the local ontology and schema files. It automates the population process by converting data retrieved from Zaragoza Open Data and DBpedia into semantic triples. The script maps entities to their respective classes and defines initial relationships, such as geographic locations and categories.

In [ ]:
from pandas.core.arrays import boolean
from rdflib import Graph, URIRef, RDF, Namespace, Literal, XSD
from rdflib.namespace import OWL, RDFS

'''
Use RDFLib to load “data.owl” as an RDF Graph. Remember to indicate that the
format of the input file is “xml”, which is how RDF Graph refers to RDF/XML
'''
#Initializing graphs
g = Graph()
g.parse("data.owl", format="xml")
s = Graph()
s.parse("schema.owl", format="xml")

#IRIs
schema_iri = str(s.value(predicate=RDF.type, object=OWL.Ontology))
ontology_iri = str(g.value(predicate=RDF.type, object=OWL.Ontology))
#Defining namespace
TOURISM = Namespace(schema_iri + "#")

#Cleaning function (preprocessing)
def clean_text(text: str) -> str:
    text = re.sub(r"^[^a-zA-ZáéíóúÁÉÍÓÚñÑ()]+|[^a-zA-ZáéíóúÁÉÍÓÚñÑ()]+$", "", text)
    return text

#Creating the URI from the preprocessed name
def make_uri(name: str) -> URIRef:
    name = clean_text(name)
    safe = name.replace(" ", "_")
    return TOURISM[safe]

#Function that adds the property triples
def add_data_properties(individual, row):
    # Check address
    if pd.notnull(row.get('hasAddress')):
        g.add((individual, TOURISM.hasAddress, Literal(row['hasAddress'], datatype=XSD.string)))

    # Check price
    if pd.notnull(row.get('hasPrice')):
        g.add((individual, TOURISM.hasPrice, Literal(row['hasPrice'], datatype=XSD.float)))

    # Check stars
    if pd.notnull(row.get('hasStars')):
        g.add((individual, TOURISM.hasStars, Literal(row['hasStars'], datatype=XSD.integer)))

    # Check forks
    if pd.notnull(row.get('hasForks')):
        g.add((individual, TOURISM.hasForks, Literal(row['hasForks'], datatype=XSD.integer)))

    # Check parking
    if pd.notnull(row.get('hasParking')):
        g.add((individual, TOURISM.hasParking, Literal(row['hasParking'], datatype=XSD.boolean)))

    # Check pet friendly
    if pd.notnull(row.get('isPetFriendly')):
        g.add((individual, TOURISM.isPetFriendly, Literal(row['isPetFriendly'], datatype=XSD.boolean)))

    # Check wifi
    if pd.notnull(row.get('hasWifi')):
        g.add((individual, TOURISM.hasWifi, Literal(row['hasWifi'], datatype=XSD.boolean)))

g.add((URIRef(ontology_iri), OWL.imports, URIRef(schema_iri)))

'''
Populate the graph with new data obtained from DBpedia, via their SPARQL
endpoints. First, use SPARQL queries to retrieve new individuals, and, second,
add new triples to the graph indicating the class that each new individual
belongs to.
'''
#Adding Zaragoza and Spain (and their connection) manually from DBPedia
DBPEDIA = "http://dbpedia.org/resource/"

zaragoza = URIRef(DBPEDIA + "Zaragoza")
spain = URIRef(DBPEDIA + "Spain")

g.add((zaragoza, RDF.type, OWL.NamedIndividual))
g.add((zaragoza, RDF.type, TOURISM.City))

g.add((spain, RDF.type, OWL.NamedIndividual))
g.add((spain, RDF.type, TOURISM.Country))

g.add((zaragoza, TOURISM.locatedInCountry, spain))

print("Done adding the initial data")


Done adding the initial data


In [ ]:
# Zaragoza Open Data
seen_uris = set() #to skip duplicates

for _, row in df.iterrows():
    name = row['name']
    category = row['category']
    accomodationType = row['accomodationType']

    ind = make_uri(name)
    if ind in seen_uris:
        continue
    seen_uris.add(ind)

    g.add((ind, RDF.type, OWL.NamedIndividual))

    if category == "parque":
        g.add((ind, RDF.type, TOURISM.Park))
        g.add((ind, TOURISM.isInCity, zaragoza))

    elif category == "museo":
        g.add((ind, RDF.type, TOURISM.Museum))
        g.add((ind, TOURISM.isInCity, zaragoza))

    elif category == "galeria":
        g.add((ind, RDF.type, TOURISM.ArtGallery))
        g.add((ind, TOURISM.isInCity, zaragoza))

    elif category == "transporte":
        if "Aeropuerto" in name:
            g.add((ind, RDF.type, TOURISM.Flight))
        else:
            g.add((ind, RDF.type, TOURISM.Transport))
        g.add((ind, TOURISM.isTransportOf, zaragoza))

    elif category == "alojamientos":
        if accomodationType == "Hotel":
            g.add((ind, RDF.type, TOURISM.Hotel))
        elif accomodationType == "Apartment":
            g.add((ind, RDF.type, TOURISM.Apartment))
        elif accomodationType == "Hostal":
            g.add((ind, RDF.type, TOURISM.Hostal))
        g.add((ind, TOURISM.isInCity, zaragoza))

    elif category == "fine_dining":
        g.add((ind, RDF.type, TOURISM.FineDining))
        g.add((ind, TOURISM.isInCity,zaragoza))

    elif category == "tradicional":
        g.add((ind, RDF.type, TOURISM.TraditionalRestaurant))
        g.add((ind, TOURISM.isInCity,zaragoza))

    elif category == "tapas":
        g.add((ind, RDF.type, TOURISM.TapasBar))
        g.add((ind, TOURISM.isInCity,zaragoza))

    elif category == "fast_food":
        g.add((ind, RDF.type, TOURISM.FastFood))
        g.add((ind, TOURISM.isInCity,zaragoza))

    elif category == "bakery":
        g.add((ind, RDF.type, TOURISM.Bakery))
        g.add((ind, TOURISM.isInCity,zaragoza))

    elif category == "restaurante":
        g.add((ind, RDF.type, TOURISM.GastronomicPlace))
        g.add((ind, TOURISM.isInCity,zaragoza))

    elif category == "monumento":
        if "romanas" in name.lower():
            g.add((ind, RDF.type, TOURISM.RomanRuins))
        elif "museo" in name.lower() or "museum" in name.lower():
            g.add((ind, RDF.type, TOURISM.Museum))
        elif "iglesia" in name.lower() or "basílica" in name.lower():
            g.add((ind, RDF.type, TOURISM.ReligiousBuilding))
        else:
            g.add((ind, RDF.type, TOURISM.CulturalPlace))
        g.add((ind, TOURISM.isInCity, zaragoza))

    add_data_properties(ind, row)

print("Done adding Zaragoza Open Data individuals")

Done adding Zaragoza Open Data individuals


In [ ]:
dbpedia_onto = {
    "PublicTransitSystem": TOURISM.Transport,
    "Castle": TOURISM.MudejarMonument,
    "Holiday": TOURISM.ReligiousEvent
}

# DBpedia data
for _, row in df_dbpedia.iterrows():
    name = row['name']
    category = row['category']
    ind = make_uri(name)
    if ind in seen_uris:
        continue
    seen_uris.add(ind)

    # Add individual and relationship to city
    g.add((ind, RDF.type, OWL.NamedIndividual))

    if category == "PublicTransitSystem":
        if "tram" in name:
            g.add((ind, RDF.type, TOURISM.Tram))
        elif "AVE" in name:
            g.add((ind, RDF.type, TOURISM.HighSpeedTrain))
        else:
            g.add((ind, RDF.type, TOURISM.Transport))
        g.add((ind, TOURISM.isTransportOf, zaragoza))

    else:
        target_class = dbpedia_onto.get(category)
        if target_class:
            g.add((ind, RDF.type, target_class))
            g.add((ind, TOURISM.isInCity, zaragoza))

print("Done adding DBpedia individuals")

Done adding DBpedia individuals


# 5. Manual Data Enrichment
This section manually adds specific entities and relationships to the knowledge graph that were not available through the SPARQL endpoints. This step helps complete the dataset and maintains the logical connections between different tourism resources in Zaragoza.

In [ ]:
# Districts in Zaragoza
districts = [
    'Centro',
    'Delicias',
    'Universidad',
    'San José',
    'Arrabal',
    'La Almozara',
    'Oliver-Valdefierro',
    'Torrero-La Paz',
    'Casco Antiguo',
    'Valdespartera',
    'Actur',
    'Las Fuentes',
    'La Jota'
  ]
for dis in districts:
    ind = make_uri(dis)
    g.add((ind, RDF.type, OWL.NamedIndividual))
    g.add((ind, RDF.type, TOURISM.District))
    g.add((ind, TOURISM.partOfCity, zaragoza))

In [ ]:
# Tourist place
tourist_place = {
    'Espacio_Expo_ZGZ': {
    'type': TOURISM.TouristPlace,
    'address': "Pablo Ruiz Picasso, 61",
    'district': "Actur"
    },
    # NightlifePlace
    'La_Terraza': {
        'type': TOURISM.Bar,
        'address': "C. de San Antonio María Claret, 20",
        'district': "Universidad"
    },
    'Loch_Ness': {
        'type': TOURISM.Pub,
        'address': "C. de Baltasar Gracián, 29",
        'district': "Universidad"
    },
    'Supernova_Club': {
        'type': TOURISM.Club,
        'address': "Av. de José Atarés, 7",
        'district': "Actur"
    },
    # NaturePlace
    'Riberas_del_Ebro': {
        'type': TOURISM.RiverBank,
    },
    # GastronomicPlace
    'Panaderia_Pasteleria_Bakery_Cakes': {
        'type': TOURISM.Bakery,
    },
    'Foodverzo': {
        'type': TOURISM.FastFood,
        'address': "C. de José Luis Albareda, 3",
        'district': "Casco Antiguo",
        'forks': 1
    },
    # CulturalPlace
    'Plaza_Espana': {
        'type': TOURISM.Square,
        'district': "Casco Antiguo"
    },
    'Plaza_Aragon': {
        'type': TOURISM.Square,
        'district': "Centro"
    },
    'Plaza_Del_Pilar':{
        'type': TOURISM.Square,
        'district': "Casco Antiguo"
    }
}

for place_name, props in tourist_place.items():
    place_uri = make_uri(place_name)
    g.add((place_uri, RDF.type, OWL.NamedIndividual))
    g.add((place_uri, RDF.type, props['type']))

    g.add((ind, RDF.type, OWL.NamedIndividual))
    g.add((ind, RDF.type, TOURISM.District))

    if 'address' in props:
        g.add((place_uri, TOURISM.hasAddress, Literal(props['address'])))

    if 'district' in props:
        dist_uri = make_uri(props['district'])
        g.add((place_uri, TOURISM.isInDistrict, dist_uri))
    else:
         g.add((place_uri, TOURISM.isInCity, zaragoza))

print("Manual mapping done")

Manual mapping done


In [ ]:
#Transport
transport = {
    'Alsa': TOURISM.Bus,
    'BiZi': TOURISM.Bicycle,
    'Radio_Taxi_Zaragoza': TOURISM.Taxi,
    'Avanza_Zaragoza': TOURISM.Bus,
}

for name, cls in transport.items():
    ind = make_uri(name)
    g.add((ind, RDF.type, OWL.NamedIndividual))
    g.add((ind, RDF.type, cls))
    g.add((ind, TOURISM.isTransportOf,zaragoza))


g.add((make_uri('BiZi'), TOURISM.connectsTo, make_uri('Avanza_Zaragoza')))
g.add((make_uri('Radio_Taxi_Zaragoza'), TOURISM.connectsTo, TOURISM.Avanza_Zaragoza))
g.add((TOURISM.Avanza_Zaragoza, TOURISM.connectsTo, make_uri('AVE')))
g.add((TOURISM.Avanza_Zaragoza, TOURISM.connectsTo, TOURISM.Zaragoza_tram))
g.add((TOURISM.Avanza_Zaragoza, TOURISM.hasPrice, Literal(1.7, datatype=XSD.float)))
g.add((TOURISM.Estación_Zaragoza_Delicias, TOURISM.isInDistrict, TOURISM.Delicias))

<Graph identifier=N2d222105c0204796a09a2799581802ee (<class 'rdflib.graph.Graph'>)>

In [ ]:
events = {
    # Concerts
    'Love_the_90s_Zaragoza': {
        'type': TOURISM.Concert,
        'startDate': "2026-09-12",
        'endDate': "2026-09-12"
    },

    # Live Shows
    'Flamenco_Night_Show': {
        'type': TOURISM.LiveShow,
        'startDate': "2026-08-21",
        'endDate': "2026-08-21"
    },

    'Comedy_Live_Show_ZGZ': {
        'type': TOURISM.LiveShow,
        'startDate': "2026-11-14",
        'endDate': "2026-11-14"
    },

    # Festival
    'Vive_Latino_2026': {
        'type': TOURISM.Festival,
        'venue': TOURISM.Espacio_Expo_ZGZ,
        'startDate': "2026-09-04",
        'endDate': "2026-09-05"
    }
}

for event_name, props in events.items():
    event_uri = make_uri(event_name)
    g.add((event_uri, RDF.type, OWL.NamedIndividual))
    g.add((event_uri, RDF.type, props['type']))

    # Venue
    if 'venue' in props:
        g.add((event_uri, TOURISM.eventLocation, TOURISM.Espacio_Expo_ZGZ))
    else:
        g.add((event_uri, TOURISM.isInCity,zaragoza))

    # Start date
    if 'startDate' in props:
        g.add((event_uri,
               TOURISM.startDate,
               Literal(props['startDate'],
                       datatype=XSD.date)))
    # End date
    if 'endDate' in props:
        g.add((event_uri,
               TOURISM.endDate,
               Literal(props['endDate'],
                       datatype=XSD.date)))

print("Concerts, live shows, and festivals added")

Concerts, live shows, and festivals added


# 6. Serialization and Verification
This step saves the enriched graph into the `data.owl` file. It also prints the instances for each class to verify that both automated and manual data were integrated correctly.

In [ ]:
'''
Save the graph using the same file and syntax (RDF/XML)
'''
g.serialize(destination="data2.owl", format="xml")
individuals = list(g.subjects(RDF.type, OWL.NamedIndividual))
print(f"data.owl saved: {len(g)} triples after population, with {len(individuals)} individuals") #printing final number of triples and individuals

data.owl saved: 10825 triples after population, with 2353 individuals


In [ ]:
def print_instances(g, class_uri, label):
    instances = list(g.subjects(RDF.type, class_uri))
    print(f"\nInstances of {label}:")
    for ind in instances:
        print(" ", ind)

# print_instances(g, TOURISM.Park,             "Park")
# print_instances(g, TOURISM.Museum,           "Museum")
# print_instances(g, TOURISM.ArtGallery,       "Art Gallery")
# print_instances(g, TOURISM.Transport,        "Transport")
print_instances(g, TOURISM.ReligiousBuilding,"Religious Building")
# print_instances(g, TOURISM.Flight,           "Airport (Flight)")
# print_instances(g, TOURISM.HighSpeedTrain,   "High Speed Train")
# print_instances(g, TOURISM.MudejarMonument,  "Mudéjar Monument")
# print_instances(g, TOURISM.District,         "District")
# print_instances(g, TOURISM.Hostal,           "Hostal")
# print_instances(g, TOURISM.CulturalPlace,      "Tourist place")


Instances of Religious Building:
  http://www.semanticweb.org/carlota/ontologies/2026/2/tourism#Iglesia_Parroquial_De_Santa_Maria_Magdalena
  http://www.semanticweb.org/carlota/ontologies/2026/2/tourism#Iglesia_De_San_Pablo
  http://www.semanticweb.org/carlota/ontologies/2026/2/tourism#Iglesia_De_San_Juan_De_Los_Panetes
  http://www.semanticweb.org/carlota/ontologies/2026/2/tourism#Basílica_De_Nuestra_Señora_Del_Pilar
  http://www.semanticweb.org/carlota/ontologies/2026/2/tourism#Barrio_De_La_Cartuja._Iglesia_De_La_Cartuja_Inmaculada_Concepción
  http://www.semanticweb.org/carlota/ontologies/2026/2/tourism#Barrio_De_Garrapinillos._Iglesia_De_San_Lorenzo
  http://www.semanticweb.org/carlota/ontologies/2026/2/tourism#Iglesia_De_Santo_Tomás_De_Villanueva_Del_Antiguo_Convento_De_Agustinos_De_La_Mantería
  http://www.semanticweb.org/carlota/ontologies/2026/2/tourism#Barrio_De_Monzalbarba._Torre_De_La_Antigua_Iglesia
  http://www.semanticweb.org/carlota/ontologies/2026/2/tourism#Iglesia_Par

# Ontology Reasoning and Consistency Check
This section uses `owlready2` to load the data and schema ontologies for logical analysis. It runs a reasoner to check the consistency of the graph and to confirm that new individuals have been correctly classified.

In [ ]:
# REASONING

''' Use owlready2 to load the ontologies (indicating “rdfxml” format). First, load the
schema ontology and, then, the data ontology.
'''

if "/content/" not in onto_path:
    onto_path.append("/content/")

onto = get_ontology("data2.owl").load(format="rdfxml")
schema = get_ontology("schema.owl").load(format="rdfxml")

namespace = onto.get_namespace(schema.base_iri)

''' Now use owlready2 to reason with the ontology: check that the file is already
consistent and that the new individuals have been correctly added to the
ontology (by showing the individuals of some of the classes which RDFLib
populated with new instances ).
'''

'''To access classes defined in schema.owl, we refer to them through the 'onto' object.
The data ontology contains the individuals that are instances of these classes. '''

#Using reasoner and printing errors if they exist
with onto:
    try:
        sync_reasoner()
    except Exception as e:
        print("\nERROR:")
        print("Type:", type(e))
        print("Message:", str(e))

#Examples of populated classes
print("\nIndividuals of Museum:")
with namespace:
    for ind in namespace.Museum.instances():
        print(ind)

print("\nIndividuals of Religious Building:")
with namespace:
    for ind in namespace.ReligiousBuilding.instances():
        print(ind)

* Owlready2 * Running HermiT...
    java -Xmx2000M -cp /usr/local/lib/python3.12/dist-packages/owlready2/hermit:/usr/local/lib/python3.12/dist-packages/owlready2/hermit/HermiT.jar org.semanticweb.HermiT.cli.CommandLine -c -O -D -I file:////tmp/tmpnfgh3ggr



Individuals of Museum:
tourism.Palacio_De_Los_Condes_De_Argillo._Museo_Pablo_Gargallo
tourism.Museo_Del_Teatro_De_Caesaraugusta
tourism.Emoz__Escuela_Museo_Origami_De_Zaragoza
tourism.Museo_De_Los_Faroles_Del_Rosario_De_Cristal
tourism.Museo_Provincial_De_Zaragoza:_Sección_De_Etnología_Y_Cerámica
tourism.Museo_Goya_-_Colección_Ibercaja
tourism.Museo_Del_Foro_De_Caesaraugusta
tourism.Museo_De_Zaragoza:_Secciones_De_Antigüedad__Y_Bellas_Artes
tourism.Museo_De_Las_Termas_Públicas_De_Caesaraugusta
tourism.Catedral_Del_Salvador_O_La_Seo_Y_Museo_De_Tapices
tourism.Alma_Mater_Museum
tourism.Antiguo_Convento_De_La_Victoria._Museo_Del_Fuego_Y_De_Los_Bomberos
tourism.Museo_Goya
tourism.Museo_De_Ciencias_Naturales_De_La_Universidad_De_Zaragoza
tourism.Museo_Del_Puerto_Fluvial_De_Caesaraugusta

Individuals of Religious Building:
tourism.Iglesia_De_La_Exaltación_De_La_Santa_Cruz
tourism.Iglesia_De_San_Antonio_De_Padua
tourism.Iglesia_Parroquial_De_San_Gil_Abad
tourism.Iglesia_Parroquial_De_Nuestra

* Owlready2 * HermiT took 15.77973222732544 seconds
* Owlready * (NB: only changes on entities loaded in Python are shown, other changes are done but not listed)
